# Notebook 25 — WU-20F full-cohort virtual preflight + synthetic network forward

**CPU ONLY. NO TRAINING. NO CLINICAL APPROVAL.** 149 image-header inventories and a synthetic, untrained nnU-Net 2.8.1 forward pass, both with `training_ready=false`. No patient voxels or lesion labels are opened by the preflight.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Verify frozen inputs and retrieve the SHA-pinned code

The review release remains blocked; this notebook never changes the WU20C reviewer worksheet or source 504/505 folders.

In [ ]:
import hashlib, json, os, subprocess, sys, urllib.request
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RAW=ROOT/'nnUNet_raw'/'Dataset505_ISLES2024_NCCT_PERFUSION'
PRE=ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLITS=PRE/'splits_final.json'
PLANS=PRE/'nnUNetPlans.json'
FOLDS=ROOT/'results'/'wu20b_full_fold_calibration'/'fold_private'
WU20E=ROOT/'results'/'wu20e_review_and_release_gate'
GATE=WU20E/'WU20E_release_gate_aggregate.json'
POLICY=WU20E/'WU20E_policy_PROPOSED_NOT_APPROVED_PRIVATE.json'
OUT=ROOT/'results'/'wu20f_virtual_staging_and_forward'
SOURCE=Path('/content/wu20f_pinned_source')
for folder in (SOURCE/'src'/'hybridstrokeseg', SOURCE/'scripts'):
    folder.mkdir(parents=True,exist_ok=True)
(SOURCE/'src'/'hybridstrokeseg'/'__init__.py').write_text('')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()
assert RAW.is_dir() and FOLDS.is_dir() and GATE.is_file() and POLICY.is_file()
assert sha256(SPLITS)=='664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
assert sha256(PLANS)=='6203521136de2f57bbe3895a639863112a5869697de9b752ade42dc1a23e4dca'
COMMIT='555ba5f38db2ff4355d6ea870a8f19f041e536b3'
BASE=f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{COMMIT}'
FILES=(
    'src/hybridstrokeseg/wu19_normalization_protocol.py',
    'src/hybridstrokeseg/wu20_fold_preprocessing.py',
    'scripts/qualify_isles2024_nnunet_runtime.py',
    'scripts/preflight_isles2024_wu20f_staging.py',
    'scripts/qualify_isles2024_wu20f_synthetic_forward.py',
)
for path in FILES:
    dest=SOURCE/path
    dest.parent.mkdir(parents=True,exist_ok=True)
    with urllib.request.urlopen(f'{BASE}/{path}',timeout=80) as resp:
        dest.write_bytes(resp.read())
    print('Retrieved pinned code:',path,'sha256=',sha256(dest))
print('Frozen sources verified; reviewer approval not implied.')


## 2. Install the exact nnU-Net release on CPU

The network forward test uses **actual nnU-Net v2.8.1**. Installation failures are fatal.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
                'nnunetv2==2.8.1','nibabel>=5.2,<6'],check=True)
import importlib.metadata
assert importlib.metadata.version('nnunetv2')=='2.8.1'
print('Verified nnunetv2',importlib.metadata.version('nnunetv2'))


## 3. Check all 149 patient five-channel NIfTI headers; generate no trainable datasets

Resumable metadata-only checks. All 745 image headers are verified; label files are **checked for presence and metadata only**. The output never contains transformed images or trained checkpoints.

In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
ENV=os.environ.copy()
ENV['PYTHONPATH']=str(SOURCE/'src')+os.pathsep+str(SOURCE/'scripts')+os.pathsep+ENV.get('PYTHONPATH','')
SCRIPT=SOURCE/'scripts'/'preflight_isles2024_wu20f_staging.py'
COMMAND=[sys.executable,'-u',str(SCRIPT),
    '--raw505',str(RAW),'--splits',str(SPLITS),'--plans',str(PLANS),
    '--folds',str(FOLDS),'--wu20e-gate',str(GATE),'--wu20e-policy',str(POLICY),
    '--output',str(OUT)]
LOG=OUT/'WU20F_preflight_CPU.log'
with LOG.open('w',encoding='utf8') as log:
    with subprocess.Popen(COMMAND,env=ENV,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
            log.write(line);log.flush()
        code=proc.wait()
if code:
    raise RuntimeError(f'WU20F preflight failed or is incomplete: exit={code}; inspect {LOG}')
INV=json.loads((OUT/'WU20F_preflight_aggregate.json').read_text())
assert INV['status']=='VIRTUAL_FULL_COHORT_PREFLIGHT_COMPLETE'
assert INV['n_source_image_headers_checked']==745 and INV['n_patients']==149
assert INV['training_ready'] is False and INV['staging_materialized'] is False
print('All header checks passed. This is NOT transformed dataset staging.')


## 4. Instantiate each locked nnU-Net architecture with synthetic CPU tensors

The models are **untrained**; a finite two-class forward only tests architecture compatibility, not prediction quality or patient inference.

In [ ]:
SCRIPT=SOURCE/'scripts'/'qualify_isles2024_wu20f_synthetic_forward.py'
COMMAND=[sys.executable,'-u',str(SCRIPT),'--source-plans',str(PLANS),'--output',str(OUT)]
result=subprocess.run(COMMAND,env=ENV,text=True,capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'WU20F synthetic CPU forward failed: exit={result.returncode}')
FORWARD=json.loads((OUT/'WU20F_synthetic_forward_aggregate.json').read_text())
assert FORWARD['status']=='SYNTHETIC_FORWARD_PASS_COMPLETE'
assert FORWARD['training_ready'] is False
assert FORWARD['trained_model_inference_performed'] is False
assert [FORWARD['arms'][a]['input_channels'] for a in ('506','507')]==[1,5]
print('WU20F synthetic forward passed; independent clinical/engineering release gates remain blocked.')


## 5. Summary

**No model weights were trained; no patient lesion labels or outcomes were evaluated.** Only after independent review and explicit sign-off can a separate physical dataset staging PR be considered.

In [ ]:
print(json.dumps({'preflight_status': INV['status'],
                  'n_cases':INV['n_patients'], 'image_headers':INV['n_source_image_headers_checked'],
                  'virtual_fold_count':len(INV['folds']),
                  'synthetic_forward_status':FORWARD['status'],
                  'model_506_parameter_count':FORWARD['arms']['506']['trainable_parameter_count'],
                  'model_507_parameter_count':FORWARD['arms']['507']['trainable_parameter_count'],
                  'training_ready':False},indent=2))
